In [19]:
import numpy as np

W, F = 12, 10
K = 3
CELL_CAP, SLICE_CAP = 2, 2
D = 5 + 3 + CELL_CAP + SLICE_CAP + 8  # = 20

N = 5000
data = []
for _ in range(N):
    sample = {
        "s":  np.random.randn(W, F).astype(np.float32),
        "a":  np.random.randn(K, D).astype(np.float32),
        "r":  float(np.random.randn()),
        "s2": np.random.randn(W, F).astype(np.float32),
        "done": bool(np.random.rand() < 0.1),
    }
    data.append(sample)

np.savez_compressed("replay_buffer.npz", data=data)
print("✅ dummy replay regenerated with D=", D)


✅ dummy replay regenerated with D= 20


In [20]:
# set F from the demo
from ain.loop.observer_rl import RLObserver, Intent
F = len(RLObserver(None, Intent(type="REDUCE_LATENCY", metric="delay_p95_ms", target=40)).features)
print("Trainer F =", F)   # should print 10 to match your demo


Trainer F = 10


In [21]:
import numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from ain.loop.model_defs import SlateDQNetwork


ACTION_TYPES = ["MCS_CAP", "PRB_WEIGHT", "SLICE_QOS", "SCHEDULER_POLICY", "REPORTING"]
SCOPES = ["CELL", "SLICE"]
CELLS = ["CELL_001", "CELL_002"]
SLICES = ["SLICE_A", "SLICE_B"]


W = 12
F = 10         
CELL_CAP  =  len(CELLS)   # or load from meta/replay
SLICE_CAP =  len(SLICES)

class QNet(nn.Module):
    def __init__(self, f_dim=F, h=1):
        super().__init__()
        self.gru = nn.GRU(input_size=f_dim, hidden_size=h, batch_first=True)
        self.act = nn.Sequential(
            nn.Linear(f_dim, h), 
            nn.ReLU(),
            nn.Linear(h, h)
        )
        self.head = nn.Sequential(
            nn.Linear(2*h, h), 
            nn.ReLU(),
            nn.Linear(h, 1)
        )
    def forward(self, s_win, a_vec):
        # s_win: [B,W,F], a_vec: [B,A_DIM]
        g, _ = self.gru(s_win)              # [B,W,H]
        s_emb = g[:, -1, :]                 # [B,H]
        a_emb = self.act(a_vec)             # [B,H]
        x = torch.cat([s_emb, a_emb], dim=1)
        return self.head(x).squeeze(1)      # [B]

class Replay(Dataset):
    def __init__(self, path):
        buf = np.load(path, allow_pickle=True)["data"]
        self.data = buf
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        t = self.data[i]   # already a dict
        return (t["s"], t["a"], t["r"], t["s2"], float(t["done"]))



def td_target(qnet, s2, a_best_next, gamma=0.95):
    with torch.no_grad():
        q_next = qnet(s2, a_best_next)
    return q_next * gamma

# Load
ds = Replay("replay_buffer.npz")
dl = DataLoader(ds, batch_size=256, shuffle=True, drop_last=True)

net = SlateDQNetwork(feat_dim=F, cell_cap=CELL_CAP, slice_cap=SLICE_CAP)
tgt = SlateDQNetwork(feat_dim=F, cell_cap=CELL_CAP, slice_cap=SLICE_CAP)
tgt.load_state_dict(net.state_dict())
opt = optim.Adam(net.parameters(), lr=1e-3)
gamma = 0.95
tau = 0.005

for epoch in range(10):
    for s, p, r, s2, done in dl:
        s  = torch.tensor(s).float()
        p  = torch.tensor(p).float()
        s2 = torch.tensor(s2).float()
        r  = torch.tensor(r).float()
        done = torch.tensor(done).float()


        q = net(s, p)

        with torch.no_grad():
            q2 = tgt(s2, p)
            y  = r + (1.0 - done) * gamma * q2

        loss = (q - y).pow(2).mean()
        opt.zero_grad(); loss.backward(); opt.step()

        with torch.no_grad():
            for tp, p_ in zip(tgt.parameters(), net.parameters()):
                tp.data.mul_(1 - tau).add_(tau * p_.data)


# Save weights
torch.save({
    "state_dict": net.state_dict(),
    "meta": {"W": W, "F": F, "cell_cap": CELL_CAP, "slice_cap": SLICE_CAP}
}, "models/qnet_offline.pt")
print("Saved mode to models/qnet_offline.pt")


C:\Users\rallo\AppData\Local\Temp\ipykernel_14664\1225773312.py:68: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  s  = torch.tensor(s).float()
C:\Users\rallo\AppData\Local\Temp\ipykernel_14664\1225773312.py:69: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  p  = torch.tensor(p).float()
C:\Users\rallo\AppData\Local\Temp\ipykernel_14664\1225773312.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  s2 = torch.tensor(s2).float()
C:\Users\rallo\AppData\Local\Temp\ipykernel_14664\1225773312.py:71: UserWarning: To copy construct from a tensor, 

Saved mode to models/qnet_offline.pt
